In [1]:
# ═══════════════════════════════════════════════════════════
# CELL 1: Install Training Stack
# ═══════════════════════════════════════════════════════════

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Unsloth — 4-bit quantized model loading + LoRA + Triton attention
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# RL + adapter stack
!pip install --no-deps trl peft accelerate bitsandbytes

# Dataset + tracking + project deps
!pip install datasets wandb pydantic>=2.0.0 matplotlib

# Verify
from unsloth import FastLanguageModel
import trl, peft, wandb
print(f"✓ Unsloth + TRL {trl.__version__} + PEFT {peft.__version__} ready")

GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-zhxcjcig/unsloth_d1128e5204bb4b1cb253ca57049377d6
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-zhxcjcig/unsloth_d1128e5204bb4b1cb253ca57049377d6
  Resolved https://github.com/unslothai/unsloth.git to commit b09aa82a3ac9ac9794c497e4b8f747f77e52b162
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 158.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 132.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# ═══════════════════════════════════════════════════════════
# CELL 2: Clone the project
# ═══════════════════════════════════════════════════════════

!git clone https://github.com/razancodes/Meta-Pytorch-Hackathon.git
%cd Meta-Pytorch-Hackathon

Cloning into 'Meta-Pytorch-Hackathon'...
remote: Enumerating objects: 851, done.
remote: Counting objects: 100% (219/219), done.
remote: Compressing objects: 100% (141/141), done.
remote: Total 851 (delta 100), reused 174 (delta 71), pack-reused 632 (from 1)
Receiving objects: 100% (851/851), 1.84 MiB | 27.28 MiB/s, done.
Resolving deltas: 100% (438/438), done.
/content/Meta-Pytorch-Hackathon


In [9]:
!git pull origin main

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 2.14 KiB | 2.14 MiB/s, done.
From https://github.com/razancodes/Meta-Pytorch-Hackathon
 * branch            main       -> FETCH_HEAD
   b0eb47f..8f96586  main       -> origin/main
Updating b0eb47f..8f96586
Fast-forward
 train_grpo.py | 297 +++++++++++++++++++++++++++++++++++++---------------------
 1 file changed, 192 insertions(+), 105 deletions(-)


In [10]:
# ═══════════════════════════════════════════════════════════
# CELL 3: Verify environment (no GPU needed)
# ═══════════════════════════════════════════════════════════

!python tests/test_smoke.py

[setup] Flushed all __pycache__ directories

TEST: Procedural Generator — All 9 Combos
  ✓ easy/structuring: alert=ALERT-2024-8359, entities=1, txns=5, GT_keys=['CUSTI0Y6']...
  ✓ medium/structuring: alert=ALERT-2025-4432, entities=3, txns=8, GT_keys=['CUSTW5UZ']...
  ✓ hard/structuring: alert=ALERT-2024-9727, entities=5, txns=12, GT_keys=['CUST877V']...
  ✓ easy/layering: alert=ALERT-2025-2891, entities=6, txns=6, GT_keys=['CUSTKL1C', 'ENT_Z78']...
  ✓ medium/layering: alert=ALERT-2025-7592, entities=8, txns=10, GT_keys=['CUST8JDP', 'ENT_Z59']...
  ✓ hard/layering: alert=ALERT-2025-5198, entities=10, txns=15, GT_keys=['CUSTAXEQ', 'ENT_Z68']...
  ✓ easy/trade_based_ml: alert=ALERT-2024-3870, entities=4, txns=16, GT_keys=['CUSTIJ2V', 'ENT_Z92']...
  ✓ medium/trade_based_ml: alert=ALERT-2025-5520, entities=6, txns=23, GT_keys=['CUSTXWQZ', 'ENT_Z14']...
  ✓ hard/trade_based_ml: alert=ALERT-2025-1436, entities=6, txns=25, GT_keys=['CUSTNZ8V', 'ENT_Z49']...
  ✓ All 9 typology/difficulty com

In [11]:
# ═══════════════════════════════════════════════════════════
# CELL 5: GRPO Training
# ═══════════════════════════════════════════════════════════


!python train_grpo.py \
  --num-prompts 150 \
  --epochs 2 \
  --lr 5e-5 \
  --output-dir checkpoints/defender-grpo-v3


Streaming output truncated to the last 5000 lines.
│ │ "parame… │ "ENT_Z9… │          │         │          │         │          │ │
│ │ {...}}u… │   }      │          │         │          │         │          │ │
│ │          │ }        │          │         │          │         │          │ │
│ │ New AML  │ ```      │          │         │          │         │          │ │
│ │ Alert    │          │          │         │          │         │          │ │
│ │ Assigne… │ This     │          │         │          │         │          │ │
│ │ - Alert  │ will     │          │         │          │         │          │ │
│ │ ID:      │ provide  │          │         │          │         │          │ │
│ │ ALERT-2… │ us with  │          │         │          │         │          │ │
│ │ -        │ more     │          │         │          │         │          │ │
│ │ Summary: │ informa… │          │         │          │         │          │ │
│ │ Eclipse  │ on the   │          │         │          │ 

In [13]:
# ═══════════════════════════════════════════════════════════
# CELL 6: Evaluate best checkpoint (9 combos)
# ═══════════════════════════════════════════════════════════

!python eval_harness.py --checkpoint checkpoints/defender-grpo-v3

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
  [+] Loading checkpoint: checkpoints/defender-grpo-v3...
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading weights: 100% 291/291 [00:01<00:00, 171.43it/s]
Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.8 patched 32 layers with 32 QKV layers, 32 O layers a

In [14]:
# ═══════════════════════════════════════════════════════════
# CELL 7: Run 1MDB demo + download AGUI replay
# ═══════════════════════════════════════════════════════════

!python demo_eval.py --model checkpoints/defender-grpo-v3

from google.colab import files
!zip -r demo_output.zip demo_output/
files.download('demo_output.zip')


════════════════════════════════════════════════════════════
  MEMEX DEMO — 1MDB Sovereign Wealth Fund Investigation
════════════════════════════════════════════════════════════

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
  Loading model: checkpoints/defender-grpo-v3...
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading weights: 100% 291/291 [00:01<00:00, 172

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
# ═══════════════════════════════════════════════════════════
# CELL 8: Save checkpoints to Google Drive
# ═══════════════════════════════════════════════════════════

import shutil, os

src = "/content/Meta-Pytorch-Hackathon/checkpoints"
dst = "/content/drive/MyDrive/memex_checkpoints_larp"

shutil.copytree(src, dst, dirs_exist_ok=True)
print("Done! Find it in your Drive → memex_checkpoints_larp/")

Done! Find it in your Drive → memex_checkpoints_larp/


In [16]:
from google.colab import userdata
from huggingface_hub import HfApi

# 1. Retrieve the token from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# 2. Pass the token to authenticate the API
api = HfApi(token=hf_token)

# 3. Create the repo if it doesn't exist yet
api.create_repo(
    repo_id="MuazTPM/defender-model",
    repo_type="model",
    exist_ok=True
)

# 4. Push the LoRA adapter
api.upload_folder(
    folder_path="checkpoints/defender-grpo-v2",
    repo_id="MuazTPM/defender-model",
    repo_type="model",
    commit_message="Defender GRPO checkpoint (Unsloth + TRL)"
)

print("Model pushed to HuggingFace Hub!")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...kpoint-150/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

  ...ckpoint-50/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

  .../checkpoint-200/scaler.pt: 100%|##########| 1.38kB / 1.38kB            

...

  ...adapter_model.safetensors:   0%|          |  546kB /  168MB            

  ...adapter_model.safetensors:   0%|          |  548kB /  168MB            

  ...adapter_model.safetensors:   0%|          |  549kB /  168MB            

  ...adapter_model.safetensors:   0%|          |  549kB /  168MB            

Model pushed to HuggingFace Hub!
